In [1]:
import torch
import numpy as np

# для colab способ 1
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/model.py', '/content/model.py')

# для colab способ 2
# from google.colab import drive
# drive.mount('/content/drive')
# sys.path.append('/content/drive/MyDrive')

from model import SASRecModel, negative_sampling_loss

Mounted at /content/drive


In [2]:
def ndcg_at_k(rel, pred, k=10):
    # метрика NDCG@k
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    # метрика Recall@k
    recall = 1.0 if rel in pred[:k] else 0.0
    return recall

In [3]:
# Тест 1 — размерность выхода модели (метод белого ящика)

model = SASRecModel(cnt_item=100, max_seq_len=10)
x = torch.randint(1, 100, (4, 10))
out = model(x)
print(out.shape)
assert out.shape == (4, 10, 101), f'Ожидалось (4, 10, 101), получили {out.shape}'
print('Тест пройден')

torch.Size([4, 10, 101])
Тест пройден


In [4]:
# Тест 2 — Causal mask (метод белого ящика)

seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print(mask)
assert mask[0, 1] == True, 'Позиция 0 не должна видеть позицию 1 (будущее)'
assert mask[1, 0] == False, 'Позиция 1 должна видеть позицию 0 (прошлое)'
assert mask[1, 2] == True, 'Позиция 1 не должна видеть позицию 2 (будущее)'
print('Тест пройден')

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
Тест пройден


In [5]:
# Тест 3 — predict_next (модульное тестирование)

model = SASRecModel(cnt_item=100, max_seq_len=10)
history = [5, 10, 15, 20]
items, scores = model.predict_next(history, top_k=5)
print(items)
print(scores)
assert len(items) == 5, f'Ожидалось 5, получили {len(items)}'
assert len(scores) == 5, f'Ожидалось 5, получили {len(scores)}'
print('Тест пройден')

[75 28  6 17 58]
[1.5969403 1.5228198 1.3899896 1.3463808 1.3236508]
Тест пройден


In [10]:
# Тест 4 — Пустая история (негативное тестирование)

model = SASRecModel(cnt_item=100, max_seq_len=10)
try:
    items, scores = model.predict_next([], top_k=10)
    assert len(items) == 10
    print(len(items))
    print(scores)
    print(items)
    print('Тест пройден')
except Exception as e:
    print(f'Ошикбка: {e}')

10
[2.9504416 2.7518754 2.6838338 1.9608881 1.8376689 1.7563937 1.7305065
 1.5716236 1.5605685 1.5475655]
[97 80 36 34 65  5 98  6 51 15]
Тест пройден


In [11]:
# Тест 5 — Метрики NDCG и Recall (модульное тестирование)

# Правильный трек на 1 месте → NDCG = 1.0
assert abs(ndcg_at_k(5, [5, 1, 2, 3, 4], k=5) - 1.0) < 0.001, 'NDCG должен быть 1.0'
# Правильного трека нет в топе → NDCG = 0
assert ndcg_at_k(99, [1, 2, 3, 4, 5], k=5) == 0.0, 'NDCG должен быть 0'
# Правильный трек есть в топе → Recall = 1
assert recall_at_k(3, [1, 2, 3, 4, 5], k=5) == 1.0, 'Recall должен быть 1'
# Правильного трека нет → Recall = 0
assert recall_at_k(99, [1, 2, 3, 4, 5], k=5) == 0.0, 'Recall должен быть 0'
print('Тест пройден')

Тест пройден


In [15]:
# Тест 6 — Функция потерь (модульное тестирование)

logits = torch.tensor([[0.5, 2.0, 0.1]])
targets = torch.tensor([1])
weights = torch.tensor([1.0])
loss = negative_sampling_loss(logits, targets, weights, cnt_item=3, neg_samples=2)
print(loss)
assert loss.item() > 0, 'Loss должен быть положительным'
print('Тест пройден')

tensor(2.9983)
Тест пройден


In [22]:
# Тест 7 — Загрузка и предсказание модели (интеграционное тестирование)

model_test = SASRecModel(cnt_item=100)
history = [1, 2, 3, 4, 5, 10, 8, 7, 4, 5]
items, scores = model_test.predict_next(history, top_k=10)
print(items)
print(scores)

assert len(items) == 10, f'predict_next отдает {len(items)} вместо 10'
assert all(0 <= i <= 100 for i in items), 'id трека вне допустимого диапазона'
assert all(isinstance(s, (float, np.floating)) for s in scores), 'скор не число'

print(f'predict_next({history}) → top-10: {items.tolist()}')
print('Тест пройден')

[ 5 34 72 10 51 36 56 47 68  9]
[2.8789172  1.6504364  1.4315623  1.4256318  1.2379624  1.122932
 1.0671691  1.052411   1.0458709  0.85322917]
predict_next([1, 2, 3, 4, 5, 10, 8, 7, 4, 5]) → top-10: [5, 34, 72, 10, 51, 36, 56, 47, 68, 9]
Тест пройден
